## TRANSFER LEARNING RAZA PERROS

In [1]:
########################################################################
########## CLASIFICACIÓN DE RAZAS DE PERROS - TRANSFER LEARNING ########
########################################################################

import os
from pathlib import Path

DATA_DIR = Path(os.environ.get("DATA_DIR", "./data"))
MODELS_DIR = Path(os.environ.get("MODELS_DIR", "./models"))
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# ============================================================
# PASO 0. RUTA DEL DATASET
# ============================================================
ruta_perros = DATA_DIR / "OXFORD" / "PERROS"

# UNA SOLA RUTA PARA EL MEJOR MODELO
ruta_modelo = MODELS_DIR / "modelo_razas_perros_TransferLearning.keras"

IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32
SEED = 42

# ============================================================
# PASO 1. CARGA DEL DATASET
# ============================================================
dataset_completo = tf.keras.utils.image_dataset_from_directory(
    ruta_perros,
    seed=SEED,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    shuffle=True
)

class_names = dataset_completo.class_names
num_classes = len(class_names)

print("Razas de perros:", class_names)
print("Número de clases:", num_classes)

# ============================================================
# PASO 2. SEPARAR TRAIN / VALIDATION / TEST
# ============================================================
total_batches = tf.data.experimental.cardinality(dataset_completo).numpy()

train_size = int(0.70 * total_batches)
val_size = int(0.15 * total_batches)
test_size = total_batches - train_size - val_size

train_dataset = dataset_completo.take(train_size)
resto_dataset = dataset_completo.skip(train_size)

val_dataset = resto_dataset.take(val_size)
test_dataset = resto_dataset.skip(val_size)

print("\nLotes totales:", total_batches)
print("Lotes train:", tf.data.experimental.cardinality(train_dataset).numpy())
print("Lotes validation:", tf.data.experimental.cardinality(val_dataset).numpy())
print("Lotes test:", tf.data.experimental.cardinality(test_dataset).numpy())

# ============================================================
# PASO 3. OPTIMIZAR DATASET
# ============================================================
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.cache().prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.cache().prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.cache().prefetch(buffer_size=AUTOTUNE)

# ============================================================
# PASO 4. DATA AUGMENTATION
# ============================================================
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.10),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10)
])

# ============================================================
# PASO 5. MODELO BASE PREENTRENADO
# ============================================================
base_model = MobileNetV2(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

# ============================================================
# PASO 6. MODELO FINAL
# ============================================================
model = Sequential([
    tf.keras.layers.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3)),
    data_augmentation,
    tf.keras.layers.Lambda(preprocess_input),
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.30),
    Dense(128, activation="relu"),
    Dropout(0.30),
    Dense(num_classes, activation="softmax")
])

# ============================================================
# PASO 7. COMPILACIÓN
# ============================================================
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# 🔥 VER ARQUITECTURA DEL MODELO
model.summary()

# ============================================================
# PASO 8. CALLBACKS
# ============================================================
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=2,
    min_lr=1e-6,
    verbose=1
)

checkpoint = ModelCheckpoint(
    filepath=ruta_modelo,
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=False,
    mode="min",
    verbose=1
)

# ============================================================
# PASO 9. ENTRENAMIENTO
# ============================================================
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=30,
    callbacks=[early_stopping, reduce_lr, checkpoint],
    verbose=1
)

# ============================================================
# PASO 9.1 CARGAR EL MEJOR MODELO
# ============================================================
print("\nCargando el mejor modelo guardado...")
model = load_model(
    ruta_modelo,
    custom_objects={"preprocess_input": preprocess_input},
    compile=False
)


# 🔥 SOLUCIÓN: recompilar modelo
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


# ============================================================
# PASO 10. EVALUACIÓN FINAL EN TEST
# ============================================================
test_loss, test_acc = model.evaluate(test_dataset, verbose=0)
print("\nTest loss:", test_loss)
print("Test accuracy:", test_acc)

# ============================================================
# PASO 11. MATRIZ DE CONFUSIÓN Y REPORTE
# ============================================================
y_true = []
y_pred = []

for images, labels in test_dataset:
    predicciones = model.predict(images, verbose=0)
    clases_predichas = np.argmax(predicciones, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(clases_predichas)

# 🔥 NUEVO
acc = accuracy_score(y_true, y_pred)
print("\nAccuracy score (sklearn):", acc)

print("\nMatriz de confusión:")
print(confusion_matrix(y_true, y_pred))

print("\nReporte de clasificación:")
print(classification_report(y_true, y_pred, target_names=class_names))

# ============================================================
# PASO 12. MODELO FINAL
# ============================================================
print("\nEl mejor modelo quedó guardado en:", ruta_modelo)

Found 4990 files belonging to 25 classes.
Razas de perros: ['American bulldog', 'American pitbull', 'Basset hound', 'Beagle', 'Boxer', 'Chihuahua', 'English cocker', 'English setter', 'German', 'Great pyrenees', 'Havanese', 'Japanese chin', 'Keeshond', 'Leonberger', 'Miniature pinscher', 'Newfoundland', 'Pomeranian', 'Pug', 'Saint bernard', 'Samoyed', 'Scottish', 'Shiba', 'Staffordshire', 'Wheaten', 'Yorkshire']
Número de clases: 25

Lotes totales: 156
Lotes train: 109
Lotes validation: 23
Lotes test: 24



Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ sequential (Sequential)              │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lambda (Lambda)                      │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ mobilenetv2_1.00_224 (Functional)    │ (None, 7, 7, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 1280)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │         163,968 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 25)                  │           3,225 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,425,177 (9.25 MB)

 Trainable params: 167,193 (653.10 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Epoch 1/30
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.1024 - loss: 3.3180
Epoch 1: val_loss improved from None to 2.07648, saving model to C:\Users\Oscar Ferreira\OneDrive - AUTO LINEAS AMERICA SA DE CV\Escritorio\MCD\5 - APRENDIZAJE PROFUNDO\PROYECTO FINAL\Modelos Mascotas\modelo_razas_perros_TransferLearning.keras

Epoch 1: finished saving model to C:\Users\Oscar Ferreira\OneDrive - AUTO LINEAS AMERICA SA DE CV\Escritorio\MCD\5 - APRENDIZAJE PROFUNDO\PROYECTO FINAL\Modelos Mascotas\modelo_razas_perros_TransferLearning.keras
109/109 ━━━━━━━━━━━━━━━━━━━━ 231s 2s/step - accuracy: 0.1789 - loss: 2.9716 - val_accuracy: 0.6399 - val_loss: 2.0765 - learning_rate: 1.0000e-04
Epoch 2/30
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.4265 - loss: 2.1454
Epoch 2: val_loss improved from 2.07648 to 1.10783, saving model to C:\Users\Oscar Ferreira\OneDrive - AUTO LINEAS AMERICA SA DE CV\Escritorio\MCD\5 - APRENDIZAJE PROFUNDO\PROYECTO FINAL\Modelos Mascotas\modelo_razas_perros_Tr

## FINE TUNING RAZA PERROS

In [5]:
########################################################################
########## CLASIFICACIÓN DE RAZAS DE PERROS - FINE TUNING ###############
########################################################################

import os
from pathlib import Path

DATA_DIR = Path(os.environ.get("DATA_DIR", "./data"))
MODELS_DIR = Path(os.environ.get("MODELS_DIR", "./models"))
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# ============================================================
# PASO 0. RUTAS
# ============================================================
ruta_perros = DATA_DIR / "OXFORD" / "PERROS"

# Modelo generado por tu script de Transfer Learning
ruta_modelo_transfer = MODELS_DIR / "modelo_razas_perros_TransferLearning.keras"

# Nuevo modelo de Fine Tuning
ruta_modelo_finetuning = MODELS_DIR / "modelo_razas_perros_FineTuning.keras"

IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32
SEED = 42

# ============================================================
# PASO 1. CARGA DEL DATASET
# ============================================================
dataset_completo = tf.keras.utils.image_dataset_from_directory(
    ruta_perros,
    seed=SEED,
    image_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    shuffle=True
)

class_names = dataset_completo.class_names
num_classes = len(class_names)

print("Razas de perros:", class_names)
print("Número de clases:", num_classes)

# ============================================================
# PASO 2. SEPARAR TRAIN / VALIDATION / TEST
# ============================================================
total_batches = tf.data.experimental.cardinality(dataset_completo).numpy()

train_size = int(0.70 * total_batches)
val_size = int(0.15 * total_batches)
test_size = total_batches - train_size - val_size

train_dataset = dataset_completo.take(train_size)
resto_dataset = dataset_completo.skip(train_size)

val_dataset = resto_dataset.take(val_size)
test_dataset = resto_dataset.skip(val_size)

print("\nLotes totales:", total_batches)
print("Lotes train:", tf.data.experimental.cardinality(train_dataset).numpy())
print("Lotes validation:", tf.data.experimental.cardinality(val_dataset).numpy())
print("Lotes test:", tf.data.experimental.cardinality(test_dataset).numpy())

# ============================================================
# PASO 3. OPTIMIZAR DATASET
# ============================================================
AUTOTUNE = tf.data.AUTOTUNE

train_dataset = train_dataset.cache().prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.cache().prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.cache().prefetch(buffer_size=AUTOTUNE)

# ============================================================
# PASO 4. CARGAR MODELO DE TRANSFER LEARNING
# ============================================================
print("\nCargando modelo base entrenado con Transfer Learning...")
model = load_model(
    ruta_modelo_transfer,
    custom_objects={"preprocess_input": preprocess_input},
    compile=False
)

# ============================================================
# PASO 5. LOCALIZAR Y DESCONGELAR EL MODELO BASE REAL
# ============================================================
base_model = None

print("\nCapas principales del modelo cargado:")
for layer in model.layers:
    print(layer.name, "->", layer.__class__.__name__)
    if "mobilenetv2" in layer.name.lower():
        base_model = layer

if base_model is None:
    raise ValueError("No se encontró MobileNetV2 dentro del modelo cargado.")

print("\nModelo base encontrado:", base_model.name)

# Descongelar MobileNetV2
base_model.trainable = True

# CONFIGURACIÓN 1 RECOMENDADA
fine_tune_at = 150

# Congelar las primeras capas y dejar entrenables solo las últimas
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

for layer in base_model.layers[fine_tune_at:]:
    layer.trainable = True

print(f"\nCapas totales en base_model: {len(base_model.layers)}")
print(f"Capas congeladas hasta: {fine_tune_at}")
print("Capas finales desbloqueadas para Fine Tuning.")

# ============================================================
# PASO 6. VERIFICAR CAPAS ENTRENABLES EN BASE MODEL
# ============================================================
entrenables = sum(1 for layer in base_model.layers if layer.trainable)
no_entrenables = sum(1 for layer in base_model.layers if not layer.trainable)

print("\n========================================")
print("RESUMEN DE CAPAS EN BASE_MODEL")
print("========================================")
print("Capas entrenables en base_model    :", entrenables)
print("Capas congeladas en base_model     :", no_entrenables)
print("Capas totales en base_model        :", len(base_model.layers))

print("\nÚltimas capas del base_model:")
ultimas = base_model.layers[-15:]
inicio = len(base_model.layers) - len(ultimas)

for i, layer in enumerate(ultimas):
    idx_real = inicio + i
    print(f"{idx_real:03d} | {layer.name:35s} | trainable={layer.trainable}")

# ============================================================
# PASO 7. RECOMPILAR CON NUEVO LEARNING RATE
# ============================================================
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# ============================================================
# PASO 8. CALLBACKS
# ============================================================
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=1,
    min_lr=1e-7,
    verbose=1
)

checkpoint = ModelCheckpoint(
    filepath=ruta_modelo_finetuning,
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=False,
    mode="min",
    verbose=1
)

# ============================================================
# PASO 9. ENTRENAMIENTO FINE TUNING
# ============================================================
history_fine = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=12,
    callbacks=[early_stopping, reduce_lr, checkpoint],
    verbose=1
)

# ============================================================
# PASO 10. CARGAR EL MEJOR MODELO DE FINE TUNING
# ============================================================
print("\nCargando el mejor modelo guardado de Fine Tuning...")
model = load_model(
    ruta_modelo_finetuning,
    custom_objects={"preprocess_input": preprocess_input},
    compile=False
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# ============================================================
# PASO 11. EVALUACIÓN FINAL EN TEST
# ============================================================
test_loss, test_acc = model.evaluate(test_dataset, verbose=0)
print("\nTest loss:", test_loss)
print("Test accuracy:", test_acc)

# ============================================================
# PASO 12. MATRIZ DE CONFUSIÓN Y REPORTE
# ============================================================
y_true = []
y_pred = []

for images, labels in test_dataset:
    predicciones = model.predict(images, verbose=0)
    clases_predichas = np.argmax(predicciones, axis=1)

    y_true.extend(labels.numpy())
    y_pred.extend(clases_predichas)

acc = accuracy_score(y_true, y_pred)
print("\nAccuracy score (sklearn):", acc)

print("\nMatriz de confusión:")
print(confusion_matrix(y_true, y_pred))

print("\nReporte de clasificación:")
print(classification_report(y_true, y_pred, target_names=class_names))

# ============================================================
# PASO 13. MODELO FINAL
# ============================================================
print("\nEl mejor modelo Fine Tuning quedó guardado en:", ruta_modelo_finetuning)

Found 4990 files belonging to 25 classes.
Razas de perros: ['American bulldog', 'American pitbull', 'Basset hound', 'Beagle', 'Boxer', 'Chihuahua', 'English cocker', 'English setter', 'German', 'Great pyrenees', 'Havanese', 'Japanese chin', 'Keeshond', 'Leonberger', 'Miniature pinscher', 'Newfoundland', 'Pomeranian', 'Pug', 'Saint bernard', 'Samoyed', 'Scottish', 'Shiba', 'Staffordshire', 'Wheaten', 'Yorkshire']
Número de clases: 25

Lotes totales: 156
Lotes train: 109
Lotes validation: 23
Lotes test: 24

Cargando modelo base entrenado con Transfer Learning...

Capas principales del modelo cargado:
sequential -> Sequential
lambda -> Lambda
mobilenetv2_1.00_224 -> Functional
global_average_pooling2d -> GlobalAveragePooling2D
dropout -> Dropout
dense -> Dense
dropout_1 -> Dropout
dense_1 -> Dense

Modelo base encontrado: mobilenetv2_1.00_224

Capas totales en base_model: 154
Capas congeladas hasta: 150
Capas finales desbloqueadas para Fine Tuning.

RESUMEN DE CAPAS EN BASE_MODEL
Capas en

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ sequential (Sequential)              │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lambda (Lambda)                      │ (None, 224, 224, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ mobilenetv2_1.00_224 (Functional)    │ (None, 7, 7, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 1280)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │         163,968 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 25)                  │           3,225 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,425,177 (9.25 MB)

 Trainable params: 579,993 (2.21 MB)

 Non-trainable params: 1,845,184 (7.04 MB)

Epoch 1/12
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 646ms/step - accuracy: 0.8743 - loss: 0.4232
Epoch 1: val_loss improved from None to 0.24553, saving model to C:\Users\Oscar Ferreira\OneDrive - AUTO LINEAS AMERICA SA DE CV\Escritorio\MCD\5 - APRENDIZAJE PROFUNDO\PROYECTO FINAL\Modelos Mascotas\modelo_razas_perros_FineTuning.keras

Epoch 1: finished saving model to C:\Users\Oscar Ferreira\OneDrive - AUTO LINEAS AMERICA SA DE CV\Escritorio\MCD\5 - APRENDIZAJE PROFUNDO\PROYECTO FINAL\Modelos Mascotas\modelo_razas_perros_FineTuning.keras
109/109 ━━━━━━━━━━━━━━━━━━━━ 91s 793ms/step - accuracy: 0.8830 - loss: 0.3961 - val_accuracy: 0.9226 - val_loss: 0.2455 - learning_rate: 1.0000e-05
Epoch 2/12
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 616ms/step - accuracy: 0.8816 - loss: 0.3829
Epoch 2: val_loss improved from 0.24553 to 0.24222, saving model to C:\Users\Oscar Ferreira\OneDrive - AUTO LINEAS AMERICA SA DE CV\Escritorio\MCD\5 - APRENDIZAJE PROFUNDO\PROYECTO FINAL\Modelos Mascotas\modelo_razas_perros_FineTu